In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('AnimeRecommender') \
    .config('spark.ui.port', '4040') \
    .getOrCreate()

print(f'Spark version: {spark.version}')
print('Spark UI: http://localhost:4040')

Spark version: 3.5.0
Spark UI: http://localhost:4040


In [2]:
BASE = '/home/jovyan/work/data/preprocessed'

reviews_data = spark.read.csv(f'{BASE}/reviews.csv', header=True, inferSchema=True)
anime_data   = spark.read.csv(f'{BASE}/animes.csv',    header=True, inferSchema=True)

In [3]:
from pyspark.ml.feature import StringIndexer
review_indexed = StringIndexer(inputCol="profile", outputCol="user_id")
reviews_data_indexed = review_indexed.fit(reviews_data).transform(reviews_data)

final_data = reviews_data_indexed.select(
    col('user_id').cast('integer'),
    col('anime_id').cast('integer'),
    col('score').cast('float')
).filter(col('score') > 0).filter( col('score') <= 10)

In [4]:
from recommenders import train_item_item

model = train_item_item(anime_data)

In [5]:
target_aid = final_data.select('anime_id').first()[0]
r = model(target_aid)
for index, row in r.iterrows():
    print(row['anime_id'])

34096
15417
28977
9735
9969
918
15335
7472
36838
35843


In [7]:
from recommenders import get_similar_items_for_user
target_uid = final_data.select('user_id').first()[0]
get_similar_items_for_user(target_uid, final_data, model, 10)

Get history: 0.723191499710083 secs!
Size of history: 10
Get candidates: 1.1550779342651367 secs!
Process results: 0.0019125938415527344 secs!


,anime_id,avg_sim,source_count,final_score
13,2133,1.0,1,1.0
9,39741,1.0,1,1.0
10,5420,1.0,1,1.0
11,2751,1.0,1,1.0
12,37987,1.0,1,1.0
52,34096,1.0,1,1.0
59,7472,1.0,1,1.0
58,15335,1.0,1,1.0
57,918,1.0,1,1.0
56,9969,1.0,1,1.0
